# [1-0] 데이터 프로젝트 — 우리 팀 분석 노트북

**함께 하는 방법**
- 한 화면을 같이 보며 **의논해서** 채웁니다. **키보드는 미션이 바뀔 때마다 돌아가며** 잡아요.
- 위에서부터 한 칸씩 ▶ 실행합니다. `___`(밑줄 세 개)가 있는 곳은 **직접 채워야** 실행돼요.
- 미션 1~4를 끝내면 기획안 4번(분석 결과)과 발표 슬라이드 ④⑤ 내용이 거의 완성됩니다.
- 끝나면 **파일 → Drive에 사본 저장**으로 저장하고, 우리 조 공유 폴더에 올립니다.

> 필터링·정렬·문자열 가공은 하지 않아요. 배운 `groupby`, 새 열, `isnull`, `dropna/fillna`만 씁니다.

## 질문 틀과 추천 짝

- **A 집단 비교**: ___별로 ___의 평균은 어떻게 다를까? → 막대그래프
- **B 가장 큰 집단**: ___ 중 어디에서 ___이(가) 가장 높을까? → 막대그래프
- **C 변화**: ___(연도·월)에 따라 ___은 어떻게 변할까? → 선그래프

| 파일 | 묶는 기준 `groupby("___")` | 계산할 열 `["___"]` | 새 열 아이디어 |
|---|---|---|---|
| students | study_level, activities, internet, higher_education, school_support | final_score, absences | `final_score - score_1` (성적 변화) |
| students_lesson | study_level, activities | final_score | — |
| biology | species, petal_size | petal_length_cm, sepal_width_cm | `petal_length_cm / petal_width_cm` |
| health | chest_pain_type, disease, bp_level, cholesterol_level, age_group, exercise_angina | max_heart_rate, cholesterol, disease_level | `max_heart_rate / (220 - age)` (예상 최대 심박수 대비) |
| environment | season, time_of_day, wind_level, day_type, station | pm25, temp_c | `pm25 / pm10` |
| economy | country, economy_type, continent, period, year | inflation_rate | `inflation_rate - 2` (물가 안정 목표 2%와의 차이) |
| retail | weekday, time_of_day, price_level, order_size, region, month | quantity, unit_price_gbp | `quantity * unit_price_gbp` (금액) |
| sports | decade, league, winning_season, world_series_win | home_runs, wins, runs_scored | `wins / games` (승률) |
| engineering | age_stage, uses_fly_ash, uses_superplasticizer, cement_level, age_day | strength_mpa | `water / cement` (물-시멘트비) |
| computer | ai_use, uses_python, career_stage, company_size, work_style, country_group | salary_usd, job_satisfaction | `salary_usd / 12` (월 소득) |
| art | classification, period_made, size_group, nationality_group, gender | height_cm, width_cm | `height_cm * width_cm` (면적) |
| tourism_food | price_level, cuisine, alcohol_service, city, user_budget | overall_rating, food_rating, service_rating | `(food_rating + service_rating) / 2` |
| administration | region, country, period, year | gov_effectiveness, rule_of_law, control_of_corruption | `(gov_effectiveness + rule_of_law) / 2` |
| astronomy | spectral_class, distance_group, hemisphere | color_index, absolute_mag, apparent_mag | `distance_pc * 3.262` (광년으로 환산) |
| movies | main_genre, decade, popularity_level, genre_count | avg_rating, rating_count | `2018 - release_year` (영화 나이) |

- 값이 많은 열(국가 37개 등)은 `result.head(10)`처럼 앞 10개만 그려도 돼요.
- `1_winter`, `3_afternoon`처럼 앞에 번호가 붙은 값은 그래프가 순서대로 나오게 한 거예요.
- 숫자 열을 계산할 열로, 글자 열을 묶는 기준으로 고르면 돼요. 그룹 열의 이름과 뜻은 `guides/`와 사이트의 "열 이름과 뜻 보기"에 있어요.
- 더 많은 주제 예시는 `guides/` 폴더의 파일별 안내("분석 아이디어" 표)에 있어요.
- 예시 노트북: `example_students.ipynb` (교육·상담 진로, 완성본)

In [ ]:
# 미션 0. 불러오기 — 우리 팀 파일 이름만 바꾸세요 (예: students.csv → health.csv)
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/daradarav86/bigdata/main/data/students.csv"
df = pd.read_csv(url)
display(df.head())

## 미션 1. 데이터 구조 확인
- 한 행은 무엇을 뜻하나요? (예: 학생 1명, 영화 1편, 팀의 한 시즌)
- 우리가 쓸 열 3~4개는? 숫자 열인지 글자 열인지 `info()`의 **Dtype**으로 확인해요. (int·float = 숫자, object = 글자)

In [ ]:
print(df.shape)       # (행 수, 열 수)
print(df.columns)     # 열 이름 (복사해서 쓰면 오타가 없어요)
df.info()             # 자료형과 비어 있지 않은 값의 개수

## 미션 2. 빈칸(결측치) 확인과 처리
**빈칸 개수를 보고, 방법 3가지 중 하나를 고른 뒤 이유를 한 줄로 적습니다.**

| 상황 | 고르는 방법 |
|---|---|
| 빈칸이 적고(약 5% 이하) 평균 계산에 크게 영향이 없다 | **그대로 두기** — groupby 평균은 빈칸을 자동으로 건너뜁니다 |
| 분석에 쓰는 열인데 빈칸 행을 빼도 괜찮다 | **제거** `dropna()` — shape로 줄어든 행 수 확인 |
| 채울 근거가 있다 | **대체** `fillna()` — 0이나 평균으로 성급히 채우지 않기 |

In [ ]:
print(df.isnull().sum())      # 열별 빈칸 개수

df_removed = df.dropna()      # 빈칸이 있는 행을 모두 제거한 표 (원본 df는 그대로)
print("원본", df.shape, "→ 제거 후", df_removed.shape)

df_use = df.copy()    # ← 분석에 쓸 표: df.copy()(그대로) / df_removed(제거) 중 선택
# 선택한 방법과 이유: ___

## 미션 3. 그룹별 분석
**우리 질문:** ___ 별로 ___ 의 평균은 어떻게 다를까?  *(위 추천표에서 짝을 고르세요)*

먼저 **묶는 기준**의 그룹 수와 **계산할 열**이 숫자인지 확인해요.

In [ ]:
# 확인 셀 — 묶는 기준 후보와 계산할 열을 점검합니다
print(df_use["___"].value_counts())   # 그룹이 몇 개, 각각 몇 명인가?
print(df_use.dtypes)                   # 계산할 열이 int·float(숫자)인가?

In [ ]:
# (선택) 새 열 만들기 — 예: df_use["승률"] = df_use["W"] / df_use["G"]
# df_use["새열"] = df_use["___"] * df_use["___"]

result = df_use.groupby("___")["___"].mean()    # 묶는 기준 → 계산할 열 → 통계
display(result)

## 미션 4. 그래프
- 막대: `kind="bar"` / 선: `kind="line"`
- Colab에서는 한글이 깨질 수 있으니 **제목과 축 이름은 영어**로 씁니다. (발표에서는 한글로 설명해요)

In [ ]:
result.plot(kind="bar")
plt.title("___")        # 예: Mean score by study time
plt.xlabel("___")       # 묶는 기준
plt.ylabel("___")       # 계산한 값
plt.show()

## 결과 읽기와 해석
아래 셀은 가장 큰 값, 가장 작은 값, 차이를 자동으로 계산해 줍니다. (`idxmax`, `idxmin`은 가장 큰·작은 값의 이름을 찾는 함수예요.)

In [ ]:
print("가장 큰 그룹:", result.idxmax(), round(result.max(), 2))
print("가장 작은 그룹:", result.idxmin(), round(result.min(), 2))
print("차이:", round(result.max() - result.min(), 2))

### 해석 쓰기 (기획안 4-5, 슬라이드 ⑤에 옮겨요)
① **사실** : ___(묶는 기준)별 ___의 평균은 ___이(가) 가장 높고(___), ___이(가) 가장 낮았다(___).
② **비교** : 두 집단의 차이는 ___로, ___배(또는 ___점) 정도였다.
③ **의미** : 이는 ___ 와(과) ___ 가 함께 움직이는 경향이 있음을 보여 준다. *(원인이라고 단정하지 않기)*

**한계** : 다만 ___ 때문에 이 결과를 모든 대상에게 일반화하기 어렵다.

**체크리스트**
- [ ] 빈칸 개수를 확인했다  - [ ] 처리 방법과 이유를 적었다
- [ ] 그룹별 통계 1개를 구했다  - [ ] 그래프 1개를 그렸다  - [ ] 해석과 한계를 썼다

## 막히면?
| 증상 | 확인할 것 |
|---|---|
| `KeyError: '열이름'` | 열 이름 오타예요. `print(df.columns)`에서 복사해 붙여 넣기 |
| 결과가 `NaN`이거나 비어 있음 | 그 열이 숫자 열인지(`dtypes`), 빈칸이 너무 많지 않은지(`isnull().sum()`) |
| 그래프가 안 그려짐 | 계산할 열이 숫자인지, `result`가 만들어졌는지 확인 (위 셀부터 순서대로 실행) |
| 막대가 너무 많음 | `result.head(10)`으로 앞 10개만 그리기 |
| 셀을 지웠더니 오류가 남 | 런타임 → 모두 실행이 아니라, 위에서부터 한 칸씩 다시 실행 |